In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import csv
import time


# ============================================================
# 1 - PATHS
# ============================================================

base_folder = r"C:\Users\PC-LAB1\Desktop\MHB\day-4"

dhaka_video = os.path.join(base_folder, "dhaka_traffic.mp4")
british_video = os.path.join(base_folder, "british_highway_traffic.mp4")

results_folder = os.path.join(base_folder, "results")
graphs_folder = os.path.join(results_folder, "graphs")

os.makedirs(results_folder, exist_ok=True)
os.makedirs(graphs_folder, exist_ok=True)


# ============================================================
# 2 - PARAMETERS
# ============================================================

MIN_AREA = 500

# Aspect ratio (width/height) range for a bounding box to be
# accepted as a "vehicle-like" object. Filters out thin/tall
# shapes such as pedestrians, poles, or noise blobs.
MIN_ASPECT_RATIO = 0.3
MAX_ASPECT_RATIO = 4.0

KERNEL_SIZE = 5

# MOG2 foreground mask values: ~127 = shadow, ~255 = real motion.
# Threshold at 200 keeps only strong-motion pixels and removes
# shadow pixels detected by detectShadows=True.
THRESHOLD = 200

HISTORY = 500

VAR_THRESHOLD = 50

DETECT_SHADOWS = True

# Set to True only when running locally with a display attached.
# On headless / server environments this MUST stay False, or
# cv2.imshow will raise an error (or hang).
SHOW_PREVIEW = False

# Window size (in frames) used to smooth the processing-FPS curve
# in Graph 2, since the raw per-frame value can be very noisy.
FPS_SMOOTHING_WINDOW = 15


# ============================================================
# 3 - FUNCTION TO GET VIDEO INFORMATION
# ============================================================

def get_video_info(video_path):

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error: Cannot open video:", video_path)
        return None

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frames / fps if fps > 0 else 0

    codec_value = int(cap.get(cv2.CAP_PROP_FOURCC))
    codec = "".join([
        chr(codec_value & 255),
        chr((codec_value >> 8) & 255),
        chr((codec_value >> 16) & 255),
        chr((codec_value >> 24) & 255)
    ])

    cap.release()

    return width, height, fps, frames, duration, codec


# ============================================================
# 4 - VIDEO INFORMATION
# ============================================================

print("\n======================================")
print("VIDEO INFORMATION")
print("======================================")

dhaka_info = get_video_info(dhaka_video)
british_info = get_video_info(british_video)

if dhaka_info is not None:
    print("\nDhaka Video")
    print("Width:", dhaka_info[0])
    print("Height:", dhaka_info[1])
    print("FPS:", dhaka_info[2])
    print("Frames:", dhaka_info[3])
    print("Duration:", round(dhaka_info[4], 2), "seconds")
    print("Codec:", dhaka_info[5])

if british_info is not None:
    print("\nBritish Highway Video")
    print("Width:", british_info[0])
    print("Height:", british_info[1])
    print("FPS:", british_info[2])
    print("Frames:", british_info[3])
    print("Duration:", round(british_info[4], 2), "seconds")
    print("Codec:", british_info[5])


# ============================================================
# 5 - HELPER: MOVING AVERAGE
# ============================================================

def moving_average(values, window):
    if len(values) == 0:
        return []
    values = np.array(values, dtype=float)
    kernel = np.ones(window) / window
    # 'same' keeps output length equal to input length
    smoothed = np.convolve(values, kernel, mode="same")
    return smoothed.tolist()


# ============================================================
# 6 - PROCESS VIDEOS
# ============================================================

def process_video(video_path, video_name):

    print("\n======================================")
    print("PROCESSING:", video_name)
    print("======================================")

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error: Cannot open:", video_path)
        return None

    # --------------------------------------------------------
    # Video Information
    # --------------------------------------------------------

    original_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    original_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_fps = cap.get(cv2.CAP_PROP_FPS)

    if original_width == 0 or original_height == 0:
        print("Error: Invalid video dimensions for:", video_path)
        cap.release()
        return None

    # --------------------------------------------------------
    # Resize
    # --------------------------------------------------------

    width = 960
    height = int(original_height * width / original_width)

    # --------------------------------------------------------
    # Background Subtractor
    # --------------------------------------------------------

    background = cv2.createBackgroundSubtractorMOG2(
        history=HISTORY,
        varThreshold=VAR_THRESHOLD,
        detectShadows=DETECT_SHADOWS
    )

    # --------------------------------------------------------
    # Morphology Kernel
    # --------------------------------------------------------

    kernel = np.ones((KERNEL_SIZE, KERNEL_SIZE), np.uint8)

    # --------------------------------------------------------
    # Output Video (with codec fallback)
    # --------------------------------------------------------

    output_path = os.path.join(results_folder, video_name + "_processed.mp4")

    out = None
    for codec_name in ("mp4v", "avc1"):
        fourcc = cv2.VideoWriter_fourcc(*codec_name)
        candidate = cv2.VideoWriter(output_path, fourcc, video_fps, (width, height))
        if candidate.isOpened():
            out = candidate
            break

    if out is None:
        # Final fallback: AVI with XVID, which is very broadly supported.
        output_path = os.path.join(results_folder, video_name + "_processed.avi")
        fourcc = cv2.VideoWriter_fourcc(*"XVID")
        out = cv2.VideoWriter(output_path, fourcc, video_fps, (width, height))

    # --------------------------------------------------------
    # CSV File
    # --------------------------------------------------------

    csv_path = os.path.join(results_folder, video_name + "_object_counts.csv")

    csv_file = open(csv_path, "w", newline="")
    csv_writer = csv.writer(csv_file)

    csv_writer.writerow([
        "Frame",
        "Time",
        "Object Count",
        "Processing Time (ms)",
        "Processing FPS"
    ])

    # --------------------------------------------------------
    # Variables
    # --------------------------------------------------------

    frame_number = 0

    object_counts = []
    processing_times = []
    processing_fps_values = []
    time_values = []

    max_objects = 0
    min_objects = None  # set on first valid frame, avoids fake 999999 default

    # ========================================================
    # MAIN LOOP
    # ========================================================

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame_number += 1

        # ----------------------------------------------------
        # Resize
        # ----------------------------------------------------

        frame = cv2.resize(frame, (width, height))
        original = frame.copy()

        # ----------------------------------------------------
        # Start Processing Timer
        # ----------------------------------------------------

        start_time = time.time()

        # ----------------------------------------------------
        # Background Subtraction
        # ----------------------------------------------------

        foreground = background.apply(frame)

        # ----------------------------------------------------
        # Threshold
        # ----------------------------------------------------

        _, binary = cv2.threshold(foreground, THRESHOLD, 255, cv2.THRESH_BINARY)

        # ----------------------------------------------------
        # Morphological Cleaning
        # ----------------------------------------------------

        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        binary = cv2.dilate(binary, kernel, iterations=2)

        # ----------------------------------------------------
        # Find Contours
        # ----------------------------------------------------

        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # ----------------------------------------------------
        # Detection Image
        # ----------------------------------------------------

        detected = frame.copy()
        object_count = 0

        # ----------------------------------------------------
        # Process Objects
        # ----------------------------------------------------

        for contour in contours:

            area = cv2.contourArea(contour)

            if area < MIN_AREA:
                continue

            x, y, w, h = cv2.boundingRect(contour)

            # --- Aspect ratio filter: reject non-vehicle-like blobs ---
            if h == 0:
                continue
            aspect_ratio = w / float(h)
            if aspect_ratio < MIN_ASPECT_RATIO or aspect_ratio > MAX_ASPECT_RATIO:
                continue

            object_count += 1

            cv2.rectangle(detected, (x, y), (x + w, y + h), (0, 255, 0), 2)

            cv2.putText(
                detected,
                "Object {}".format(object_count),
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 0),
                2
            )

            cv2.putText(
                detected,
                "Area: {}".format(int(area)),
                (x, y + h + 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 255),
                2
            )

        # ----------------------------------------------------
        # End Processing Timer
        # ----------------------------------------------------

        end_time = time.time()
        processing_time = (end_time - start_time) * 1000
        processing_fps = (1000 / processing_time) if processing_time > 0 else 0

        # ----------------------------------------------------
        # Save Measurements
        # ----------------------------------------------------

        current_time = (frame_number / video_fps) if video_fps > 0 else 0

        object_counts.append(object_count)
        processing_times.append(processing_time)
        processing_fps_values.append(processing_fps)
        time_values.append(current_time)

        # ----------------------------------------------------
        # Maximum / Minimum
        # ----------------------------------------------------

        if object_count > max_objects:
            max_objects = object_count

        if min_objects is None or object_count < min_objects:
            min_objects = object_count

        # ----------------------------------------------------
        # CSV
        # ----------------------------------------------------

        csv_writer.writerow([
            frame_number,
            round(current_time, 3),
            object_count,
            round(processing_time, 3),
            round(processing_fps, 2)
        ])

        # ====================================================
        # DISPLAY INFORMATION (drawn on saved output video)
        # ====================================================

        cv2.putText(detected, "TRAFFIC ANALYSIS", (20, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        cv2.putText(detected, "Video FPS: {:.2f}".format(video_fps), (20, 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 0, 0), 2)

        cv2.putText(detected, "Processing FPS: {:.2f}".format(processing_fps), (20, 100),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)

        cv2.putText(detected, "Processing Time: {:.2f} ms".format(processing_time), (20, 130),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 255), 2)

        cv2.putText(detected, "Frame: {}".format(frame_number), (20, 160),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)

        cv2.putText(detected, "Moving Objects: {}".format(object_count), (20, 190),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2)

        # ----------------------------------------------------
        # Save Processed Video
        # ----------------------------------------------------

        out.write(detected)

        # ====================================================
        # OPTIONAL LIVE PREVIEW (local runs only)
        # ====================================================

        if SHOW_PREVIEW:

            display_width = 480
            display_height = 300

            original_small = cv2.resize(original, (display_width, display_height))
            mask_small = cv2.resize(binary, (display_width, display_height))
            mask_small = cv2.cvtColor(mask_small, cv2.COLOR_GRAY2BGR)
            detected_small = cv2.resize(detected, (display_width, display_height))

            cv2.putText(original_small, "Original", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
            cv2.putText(mask_small, "Foreground Mask", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
            cv2.putText(detected_small, "Detection", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

            info_panel = np.zeros((display_height, display_width, 3), dtype=np.uint8)

            cv2.putText(info_panel, "TRAFFIC ANALYSIS", (20, 45),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
            cv2.putText(info_panel, "Video FPS: {:.2f}".format(video_fps), (20, 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(info_panel, "Processing FPS: {:.2f}".format(processing_fps), (20, 125),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(info_panel, "Processing Time: {:.2f} ms".format(processing_time), (20, 160),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(info_panel, "Frame: {}".format(frame_number), (20, 195),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(info_panel, "Moving Objects: {}".format(object_count), (20, 230),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.putText(info_panel, "Press Q to stop", (20, 270),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            top_row = np.hstack((original_small, mask_small))
            bottom_row = np.hstack((detected_small, info_panel))
            combined = np.vstack((top_row, bottom_row))

            cv2.imshow("Day 04 - Traffic Video Analysis", combined)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q") or key == 27:
                break

    # ========================================================
    # END
    # ========================================================

    cap.release()
    out.release()
    csv_file.close()

    if SHOW_PREVIEW:
        cv2.destroyAllWindows()

    # --------------------------------------------------------
    # Calculate Statistics
    # --------------------------------------------------------

    total_frames = len(object_counts)

    average_object_count = (sum(object_counts) / total_frames) if total_frames > 0 else 0
    average_processing_time = (sum(processing_times) / total_frames) if total_frames > 0 else 0
    average_processing_fps = (sum(processing_fps_values) / total_frames) if total_frames > 0 else 0

    if min_objects is None:
        min_objects = 0

    # --------------------------------------------------------
    # Print Results
    # --------------------------------------------------------

    print("\nResults for:", video_name)
    print("--------------------------------------")
    print("Video FPS:", round(video_fps, 2))
    print("Processing FPS:", round(average_processing_fps, 2))
    print("Average Object Count:", round(average_object_count, 2))
    print("Maximum Object Count:", max_objects)
    print("Minimum Object Count:", min_objects)
    print("Average Processing Time:", round(average_processing_time, 2), "ms")
    print("Output Video:", output_path)
    print("CSV:", csv_path)

    if total_frames == 0:
        print("Warning: no frames were processed, skipping graphs.")
        return {
            "video_fps": video_fps,
            "processing_fps": average_processing_fps,
            "average_objects": average_object_count,
            "max_objects": max_objects,
            "min_objects": min_objects,
            "average_time": average_processing_time
        }

    # ========================================================
    # GRAPH 1 - OBJECT COUNT VS TIME
    # ========================================================

    plt.figure(figsize=(10, 5))
    plt.plot(time_values, object_counts)
    plt.xlabel("Time (seconds)")
    plt.ylabel("Object Count")
    plt.title(video_name + " - Object Count vs Time")
    plt.grid(True)

    graph_path = os.path.join(graphs_folder, video_name + "_object_count.png")
    plt.savefig(graph_path, dpi=150, bbox_inches="tight")
    plt.close()

    # ========================================================
    # GRAPH 2 - PROCESSING FPS (raw + smoothed)
    # ========================================================

    smoothed_fps = moving_average(processing_fps_values, FPS_SMOOTHING_WINDOW)

    plt.figure(figsize=(10, 5))
    plt.plot(time_values, processing_fps_values, alpha=0.35, label="Raw")
    plt.plot(time_values, smoothed_fps, linewidth=2, label="Smoothed (window={})".format(FPS_SMOOTHING_WINDOW))
    plt.xlabel("Time (seconds)")
    plt.ylabel("Processing FPS")
    plt.title(video_name + " - Processing FPS")
    plt.legend()
    plt.grid(True)

    fps_graph_path = os.path.join(graphs_folder, video_name + "_processing_fps.png")
    plt.savefig(fps_graph_path, dpi=150, bbox_inches="tight")
    plt.close()

    return {
        "video_fps": video_fps,
        "processing_fps": average_processing_fps,
        "average_objects": average_object_count,
        "max_objects": max_objects,
        "min_objects": min_objects,
        "average_time": average_processing_time
    }


# ============================================================
# 7 - RUN DHAKA
# ============================================================

dhaka_results = process_video(dhaka_video, "dhaka")


# ============================================================
# 8 - RUN BRITISH HIGHWAY
# ============================================================

british_results = process_video(british_video, "british_highway")


# ============================================================
# 9 - FINAL COMPARISON
# ============================================================

if dhaka_results is not None and british_results is not None:

    print("\n")
    print("==============================================")
    print("GENERALIZATION EXPERIMENT")
    print("==============================================")
    print("Note: same MOG2 parameters (history, varThreshold) were")
    print("used for both videos. Traffic density/speed differ between")
    print("the two scenes, so this is a same-settings comparison,")
    print("not a fully tuned per-scene comparison.")

    print("\nMeasurement".ljust(30), "Dhaka".ljust(15), "British Highway")
    print("-" * 65)

    print("Video FPS".ljust(30),
          "{:.2f}".format(dhaka_results["video_fps"]).ljust(15),
          "{:.2f}".format(british_results["video_fps"]))

    print("Processing FPS".ljust(30),
          "{:.2f}".format(dhaka_results["processing_fps"]).ljust(15),
          "{:.2f}".format(british_results["processing_fps"]))

    print("Average Object Count".ljust(30),
          "{:.2f}".format(dhaka_results["average_objects"]).ljust(15),
          "{:.2f}".format(british_results["average_objects"]))

    print("Maximum Object Count".ljust(30),
          str(dhaka_results["max_objects"]).ljust(15),
          str(british_results["max_objects"]))

    print("Minimum Object Count".ljust(30),
          str(dhaka_results["min_objects"]).ljust(15),
          str(british_results["min_objects"]))

    print("Average Processing Time".ljust(30),
          "{:.2f} ms".format(dhaka_results["average_time"]).ljust(15),
          "{:.2f} ms".format(british_results["average_time"]))

    # ========================================================
    # COMPARISON GRAPH
    # ========================================================

    names = ["Dhaka", "British Highway"]
    fps_values = [dhaka_results["processing_fps"], british_results["processing_fps"]]

    plt.figure(figsize=(8, 5))
    plt.bar(names, fps_values)
    plt.ylabel("Processing FPS")
    plt.title("Processing FPS Comparison")
    plt.grid(axis="y")

    comparison_graph = os.path.join(graphs_folder, "processing_fps_comparison.png")
    plt.savefig(comparison_graph, dpi=150, bbox_inches="tight")
    plt.close()

print("\n======================================")
print("PROJECT FINISHED")
print("======================================")
print("Results saved in:")
print(results_folder)


VIDEO INFORMATION

Dhaka Video
Width: 640
Height: 360
FPS: 29.97002997002997
Frames: 3794
Duration: 126.59 seconds
Codec: h264

British Highway Video
Width: 1364
Height: 768
FPS: 25.0
Frames: 1549
Duration: 61.96 seconds
Codec: FMP4

PROCESSING: dhaka

Results for: dhaka
--------------------------------------
Video FPS: 29.97
Processing FPS: 162.53
Average Object Count: 16.83
Maximum Object Count: 44
Minimum Object Count: 0
Average Processing Time: 6.56 ms
Output Video: C:\Users\PC-LAB1\Desktop\MHB\day-4\results\dhaka_processed.mp4
CSV: C:\Users\PC-LAB1\Desktop\MHB\day-4\results\dhaka_object_counts.csv

PROCESSING: british_highway

Results for: british_highway
--------------------------------------
Video FPS: 25.0
Processing FPS: 132.81
Average Object Count: 8.96
Maximum Object Count: 17
Minimum Object Count: 0
Average Processing Time: 7.58 ms
Output Video: C:\Users\PC-LAB1\Desktop\MHB\day-4\results\british_highway_processed.mp4
CSV: C:\Users\PC-LAB1\Desktop\MHB\day-4\results\british_